# ARIMA Modeling and Analysis on London Smart Meter Time Series Data

This notebook demonstrates ARIMA modeling for time series forecasting using London smart meter data. The workflow includes data preparation, visualization, stationarity checks, model fitting, forecasting, residual analysis, and evaluation.

## 1. Import Required Libraries

Import all necessary libraries for data manipulation, visualization, and time series modeling, including polars, pandas, plotly, seaborn, statsforecast, statsmodels, and custom plotting utilities.

In [ ]:
# Enable autoreload for development
%load_ext autoreload
%autoreload 2

In [ ]:
# Import libraries
import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ARIMA
from statsforecast.arima import ndiffs, nsdiffs

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean

from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate
from statsmodels.tsa.stattools import adfuller

from plotly.subplots import make_subplots

from plotting_utils import (
    plot_acf,
    plot_pacf,
    plot_acf_pacf,
    plotly_series as plot_series,
    plot_series_acf_pacf,
    plot_residuals_diagnostic,
)

from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import kpss

from summary_utils import print_arima_fitted_summary, arima_fitted_summary_dataframe
from functools import partial

## 2. Load and Prepare Data

Read the preprocessed parquet file, generate datetime ranges, join and rename columns, and preview the data.

In [ ]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

## 3. Select and Filter Time Series

Filter the dataset for a specific meter ID, handle missing values, and optionally restrict the time range.

In [ ]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select([time_, id_, target_])
    .explode([time_, target_])
)
data.head()

In [ ]:
selected_id = "MAC000193"
data = (
    data.filter(id_col.eq(selected_id)).with_columns(
        target_col.forward_fill().backward_fill()
    )
    # Optionally restrict time range:
    # .filter(
    #     time_col.is_between(
    #         pl.date(year=2012, month=1, day=1), pl.date(year=2012, month=12, day=31)
    #     )
    # )
)
data.head()

## 4. Visualize Time Series

Plot the selected time series over a specified date range to inspect trends and seasonality.

In [ ]:
fig = plot_series(data, date_range=["2012-11-4", "2012-12-4"])
fig.show()

## 5. Difference the Series for Stationarity

Stationarity is a foundational concept in time series analysis, and understanding how to test for it is crucial for building reliable forecasting models like ARIMA. In this section, we’ll explore what stationarity means, why it matters, and how to formally test for it using statistical methods.

### What is Stationarity?

A time series is **stationary** if its statistical properties—such as mean, variance, and autocorrelation—do not change over time. In other words, the process generating the data behaves consistently, regardless of when you observe it.

There are two main types of stationarity:

#### a) Strict Stationarity

A time series $\{Y_t\}$ is **strictly stationary** if the joint probability distribution of any collection of values is invariant to time shifts. That is, for any set of time points $t_1, t_2, \ldots, t_k$ and any time shift $h$,

$$
(Y_{t_1}, Y_{t_2}, \ldots, Y_{t_k}) \overset{d}{=} (Y_{t_1+h}, Y_{t_2+h}, \ldots, Y_{t_k+h})
$$

where $\overset{d}{=}$ denotes equality in distribution.

- **In simpler terms:** All statistical properties (mean, variance, skewness, kurtosis, and all joint moments) are constant over time.
- **Challenge:** Strict stationarity is very strong and almost impossible to verify from a single observed time series.

#### b) Weak-Sense Stationarity (Covariance Stationarity)

A time series $\{Y_t\}$ is **weakly stationary** if it satisfies the following three conditions:

1. **Constant Mean:**  
    $$
    \mathbb{E}[Y_t] = \mu \quad \text{for all } t
    $$
    The mean $\mu$ does not depend on time.

2. **Constant Variance:**  
    $$
    \mathrm{Var}(Y_t) = \mathbb{E}[(Y_t - \mu)^2] = \sigma^2 \quad \text{for all } t
    $$
    The variance $\sigma^2$ does not depend on time.

3. **Constant Autocovariance:**  
    $$
    \mathrm{Cov}(Y_t, Y_{t-k}) = \mathbb{E}[(Y_t - \mu)(Y_{t-k} - \mu)] = \gamma_k \quad \text{for all } t \text{ and any lag } k
    $$
    The autocovariance $\gamma_k$ depends only on the lag $k$, not on the specific time $t$.

- **In simpler terms:**
     - The series hovers around a consistent average value.
     - The typical spread or fluctuation around this average is consistent.
     - The relationship (covariance) between values at two time points depends only on how far apart they are (the lag), not on when in time they occur.

> In practice, "stationarity" in ARIMA modeling almost always refers to weak stationarity.

### Why Do We Care About Stationarity?

Stationarity is crucial for several reasons, especially in the context of classical time series models like ARIMA:

- **Predictability and Inference:**  
    - If a series is stationary, the statistical properties (like mean, variance, autocorrelation) learned from one part of the series are relevant for other parts, including the future. This allows us to make meaningful inferences and build predictive models.
    - If a series is non-stationary, its properties are changing. What you learned from the past might not apply to the future, making reliable forecasting difficult. It's like trying to predict the path of a ball whose speed and direction are constantly and erratically changing.

- **Model Assumptions:**  
    - Many standard time series models (including ARMA, which is the AR and MA part of ARIMA) are designed for stationary data. Their mathematical properties and estimation procedures rely on the assumptions of constant mean, variance, and autocovariance.
    - Applying these models directly to non-stationary data can lead to:
        - *Spurious Regressions*: Finding statistically significant relationships between variables that are actually unrelated, simply because both have trends.
        - *Unreliable Parameter Estimates*: Model coefficients might not be stable or meaningful.
        - *Poor Forecasts*: The model will likely not generalize well to future data.

- **Simplification:**  
    - Working with stationary series simplifies the modeling process. We can focus on capturing the structure of the dependencies (autocorrelations) without having to simultaneously model changing means or variances.

### How Do We Achieve Stationarity?

To make a non-stationary series stationary, we often apply transformations such as:

- **Differencing:** Subtracting the previous value from the current value to remove trends or seasonality.
- **Detrending:** Removing a fitted trend line from the series.
- **Deseasonalizing:** Removing seasonal effects by differencing at the seasonal period.

As we've seen in other notebooks, electricity load data typically exhibits clear patterns of seasonality (such as daily and weekly cycles) but does not show a long-term trend. Therefore, to achieve stationarity, we focus on **seasonal differencing**—subtracting the value from the same time in the previous season (e.g., the same time yesterday or last week)—to effectively remove these repeating seasonal patterns while preserving the overall level of the series. This approach is particularly appropriate for electricity load, where the main source of non-stationarity is seasonality rather than trend.

In [ ]:
# We difference the series at daily and weekly intervals to remove trend and seasonality, making the series more suitable for ARIMA modeling and other statistical analyses.
original = data.get_column(target_)
diff_df = data.with_columns(target_col.diff(1)).drop_nulls()
diff = diff_df.get_column(target_)
diff_day_diff_df = diff_df.with_columns(target_col.diff(48)).drop_nulls()
diff_day_diff = diff_day_diff_df.get_column(target_)
day_diff_df = data.with_columns(target_col.diff(48)).drop_nulls()
day_diff = day_diff_df.get_column(target_)
week_diff_df = data.with_columns(target_col.diff(336)).drop_nulls()
week_diff = week_diff_df.get_column(target_)
day_week_diff_df = day_diff_df.with_columns(target_col.diff(336)).drop_nulls()
day_week_diff = day_week_diff_df.get_column(target_)

## Understanding Differencing in Time Series Analysis

Differencing is a fundamental technique in time series analysis, especially when preparing data for models like ARIMA. It helps transform a non-stationary series (one whose statistical properties change over time) into a stationary one, which is a key assumption for many forecasting models.

---

### 1. What is Differencing?

**Differencing** means subtracting the previous value (or a value from a previous period) from the current value in a time series. The goal is to remove trends or seasonality, making the series' mean and variance more stable over time.

#### **First-Order Differencing**

The most basic form is **first-order differencing**, which removes linear trends:

$$
y'_t = y_t - y_{t-1}
$$

where:
- $y_t$ is the value at time $t$
- $y_{t-1}$ is the value at time $t-1$
- $y'_t$ is the differenced value

If the original series has a trend, first-order differencing often makes it stationary.

---

### 2. Why Do We Difference?

- **Stationarity:** Many statistical models (like ARIMA) require the data to be stationary. Differencing helps achieve this by removing trends and seasonality.
- **Stabilizing Mean:** Differencing removes changes in the level of a time series, making the mean constant over time.
- **Removing Seasonality:** Seasonal differencing can remove repeating patterns (like daily or weekly cycles).

---

### 3. Types of Differencing

#### **a. Regular (Non-Seasonal) Differencing**

Removes trends:

$$
y'_t = y_t - y_{t-1}
$$

#### **b. Seasonal Differencing**

Removes seasonality by subtracting the value from the same season in the previous cycle:

$$
y''_t = y_t - y_{t-s}
$$

where $s$ is the season length (e.g., $s=48$ for daily seasonality in half-hourly data, $s=336$ for weekly).

---

### 4. How Many Times Should We Difference?

#### **a. Non-Seasonal Differencing Order ($d$)**

- **$d=0$:** No differencing (series is already stationary).
- **$d=1$:** First difference (removes linear trend).
- **$d=2$:** Second difference (removes quadratic trend), rarely needed.

#### **b. Seasonal Differencing Order ($D$)**

- **$D=0$:** No seasonal differencing.
- **$D=1$:** One seasonal difference (removes seasonal pattern).
- **$D=2$:** Two seasonal differences, very rarely needed.

**In practice:**  
- Most series need at most one regular and one seasonal difference.
- Over-differencing can introduce unnecessary noise and make the series harder to model.

---

### 5. Does It Make Sense to Difference Daily ($s=48$) Then Weekly ($s=336$)?

#### **a. The Logic**

- **Daily differencing ($s=48$):** Removes daily seasonality (e.g., similar patterns every day).
- **Weekly differencing ($s=336$):** Removes weekly seasonality (e.g., similar patterns every week).

#### **b. Combined Differencing**

You can apply both, but the order matters:

1. **First, daily difference:**
    $$
    y'_t = y_t - y_{t-48}
    $$
2. **Then, weekly difference on the result:**
    $$
    y''_t = y'_t - y'_{t-336}
    $$

This is equivalent to:

$$
y''_t = (y_t - y_{t-48}) - (y_{t-336} - y_{t-384}) = y_t - y_{t-48} - y_{t-336} + y_{t-384}
$$

#### **c. When Is This Useful?**

- If your data has **both strong daily and weekly seasonality**, double differencing can help.
- However, most ARIMA models only use one seasonal difference (with $s$ set to the dominant seasonality).
- **Over-differencing** can make the series too noisy and may remove useful information.

#### **d. How to Decide?**

- **Plot the series** and its autocorrelation function (ACF) to see if seasonality remains after one differencing.
- Use statistical tests (like KPSS, ADF) to check for stationarity after each differencing step.
- Use functions like `ndiffs()` and `nsdiffs()` to estimate the required number of differences.

---

### 6. Practical Example

Suppose you have half-hourly electricity data:

- **Daily seasonality:** $s=48$ (48 half-hours in a day)
- **Weekly seasonality:** $s=336$ (48 half-hours × 7 days)

**Step 1:** Daily difference

$$
y'_t = y_t - y_{t-48}
$$

**Step 2:** Weekly difference on the daily-differenced series

$$
y''_t = y'_t - y'_{t-336}
$$

---

### 7. Key Takeaways

- **Differencing is essential** for making time series stationary.
- **Start with one regular and/or one seasonal difference**; only add more if necessary.
- **Double seasonal differencing** (e.g., daily then weekly) is rare but can be justified if both seasonalities are strong and visible in the data.
- **Always check plots and stationarity tests** after each differencing step to avoid over-differencing.

---

### 8. Summary Table

| Differencing Type      | Formula                          | Purpose                |
|-----------------------|----------------------------------|------------------------|
| Regular (order $d$)   | $y_t - y_{t-1}$                  | Remove trend           |
| Seasonal (order $D$)  | $y_t - y_{t-s}$                  | Remove seasonality     |
| Double Seasonal       | $(y_t - y_{t-s_1}) - (y_{t-s_2} - y_{t-s_1-s_2})$ | Remove multiple seasonalities |

---

### 9. References

- Hyndman, R.J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*.
- Box, G.E.P., Jenkins, G.M., Reinsel, G.C., & Ljung, G.M. (2015). *Time Series Analysis: Forecasting and Control*.

---

By understanding and applying differencing thoughtfully, you can prepare your time series data for accurate and reliable forecasting!

In [ ]:
plot_series(diff_df, date_range=["2012-11-4", "2012-12-4"], title="48h difference")

In [ ]:
plot_series(
    diff_day_diff_df, date_range=["2012-11-4", "2012-12-4"], title="48h difference"
)

In [ ]:
plot_series(day_diff_df, date_range=["2012-11-4", "2012-12-4"], title="48h difference")

In [ ]:
plot_series(
    week_diff_df, date_range=["2012-11-4", "2012-12-4"], title="Weekly difference"
)

In [ ]:
plot_series(
    day_week_diff_df,
    date_range=["2012-11-4", "2012-12-4"],
    title="Daily and Weekly difference",
)

Even after applying daily, weekly, and combined daily+weekly differencing, the time series still exhibits visible seasonality and does not appear fully stationary.

This is common in electricity load data, which often contains complex or multiple seasonal patterns (e.g., daily, weekly, and possibly annual effects) and non-linearities that simple differencing cannot entirely remove. Residual seasonality may also arise from imperfect alignment of seasonal cycles, holiday effects, or interactions between different seasonalities (such as weekends vs. weekdays).

As a result, further transformations, more advanced models (like TBATS or Prophet), or the inclusion of additional exogenous variables may be necessary to fully capture and remove all non-stationary components. Always inspect both the plots and statistical tests to assess stationarity after each transformation.

### Why Does Seasonality Sometimes Remain After Differencing?

#### 3. **Why Might Seasonality Remain After Differencing?**

##### **A. Multiple Seasonalities**

Your data may have **more than one seasonal pattern**. For example, both daily and weekly cycles:

- **Daily:** $y_t \approx f_{\text{daily}}(t)$
- **Weekly:** $y_t \approx f_{\text{weekly}}(t)$

If you only difference at one period (say, daily), the other (weekly) seasonality remains.

**Example:**
- Apply daily differencing ($s=48$):
    $$
    y'_t = y_t - y_{t-48}
    $$
    This removes the daily pattern, but the weekly pattern (every 336 steps) is still present.

##### **B. Imperfect or Non-integer Seasonality**

Sometimes, the seasonal pattern is not perfectly aligned with your chosen period, or it drifts over time (e.g., due to daylight saving time, holidays, or irregular work weeks). Differencing at a fixed $s$ cannot fully remove such patterns.

##### **C. Overlapping or Interacting Seasonalities**

If the seasonalities interact (e.g., weekends have different daily patterns than weekdays), simple differencing at one period may not capture these interactions.

##### **D. Incomplete Differencing**

If you only apply regular differencing (to remove trend), but not seasonal differencing, the seasonality will remain. Conversely, if you only apply one seasonal difference but your data has two strong seasonalities, one will remain.

---

#### 4. **Mathematical Illustration**

Suppose your time series is:
$$
y_t = T_t + S^{(1)}_t + S^{(2)}_t + \varepsilon_t
$$
where:
- $T_t$ is the trend,
- $S^{(1)}_t$ is daily seasonality ($s_1=48$),
- $S^{(2)}_t$ is weekly seasonality ($s_2=336$),
- $\varepsilon_t$ is noise.

**Apply daily differencing:**
$$
y'_t = y_t - y_{t-48} = [T_t - T_{t-48}] + [S^{(1)}_t - S^{(1)}_{t-48}] + [S^{(2)}_t - S^{(2)}_{t-48}] + [\varepsilon_t - \varepsilon_{t-48}]
$$

- $[S^{(1)}_t - S^{(1)}_{t-48}] = 0$ (since $S^{(1)}$ repeats every 48)
- $[S^{(2)}_t - S^{(2)}_{t-48}] \neq 0$ (since $S^{(2)}$ repeats every 336, not 48)

So, **the weekly seasonality remains** after daily differencing.

---

#### 5. **How to Remove All Seasonality?**

- **Apply multiple seasonal differences:**
    1. Daily: $y'_t = y_t - y_{t-48}$
    2. Weekly: $y''_t = y'_t - y'_{t-336}$

- **Further transformations**, **more advanced models** (like TBATS or Prophet), or the **inclusion of additional exogenous variables** (like SARIMAX) may be necessary to fully capture and remove all non-stationary components. Always inspect both the plots and statistical tests to assess stationarity after each transformation.

---

#### 6. **Visual Inspection**

After differencing, plot your series and its autocorrelation function (ACF):

- If you see spikes at multiples of a period (e.g., every 48 or 336 lags), that seasonality is still present.
- If the ACF decays slowly at those lags, more differencing or a more complex model may be needed.

---

#### 7. **Summary Table**

| Differencing Applied | Seasonality Removed | Seasonality Remaining |
|---------------------|--------------------|----------------------|
| None                | None               | Daily, Weekly        |
| Daily ($s=48$)      | Daily              | Weekly               |
| Weekly ($s=336$)    | Weekly             | Daily                |
| Daily + Weekly      | Daily, Weekly      | (Should be none)     |

---

#### 8. **Key Takeaways**

- **Differencing at one period only removes that specific seasonality.**
- **Multiple seasonalities require multiple differencing steps.**
- **Always inspect your data visually and with ACF/PACF plots after each transformation.**
- **Some complex seasonal patterns may require advanced models, not just differencing.**

---

By understanding the structure of your data and the mathematics of differencing, you can better diagnose why seasonality remains and how to address it for effective time series modeling!

## 6. Autocorrelation and Stationarity Tests
## What Does Non-Stationarity Look Like? And How to Detect It?

Non-stationarity typically manifests in a few common ways:

- **Trend:** The mean of the series shows a long-term increase or decrease.  
    *Example:* Stock prices generally trending upwards over years, global temperatures increasing.

- **Seasonality:** The series exhibits regular, predictable fluctuations that repeat over a fixed period (e.g., daily, monthly, yearly).  
    *Example:* Ice cream sales peaking in summer, electricity demand peaking in the morning and evening.

- **Changing Variance (Heteroscedasticity):** The spread or volatility of the series changes over time.  
    *Example:* Financial returns often show periods of high volatility followed by periods of low volatility.

- **Structural Breaks:** Sudden shifts in the level or behavior of the series due to external events.  
    *Example:* A sudden drop in sales after a new competitor enters the market.

---

### Detecting Stationarity

We use a combination of visual inspection, summary statistics, statistical tests, and autocorrelation plots.

#### A. Visual Inspection

- **Plot the Time Series:**  
    Look for:
    - Obvious upward or downward trends.
    - Repeating cyclical patterns (seasonality).
    - Widening or narrowing of fluctuations (changing variance).
    - Sudden jumps or drops.

A stationary series should look roughly horizontal, with constant variance, and no obvious predictable patterns like strong seasonality.

#### B. Summary Statistics

- **Split and Compare:**  
    Divide your time series into two or more parts and calculate the mean and variance for each part. If these values are substantially different, it's an indicator of non-stationarity. This is a crude but quick check.

#### C. Statistical Tests (Hypothesis Tests)

These provide a more formal way to assess stationarity.

- **Augmented Dickey-Fuller (ADF) Test:**
    - Tests for a "unit root," a common cause of non-stationarity (specifically, trend non-stationarity). A series with a unit root is often called a "random walk."
    - Null Hypothesis ($H_0$): The series has a unit root (it is non-stationary).
    - Alternative Hypothesis ($H_a$): The series does not have a unit root (it is stationary, or trend-stationary if a trend term is included).
    - **Interpretation:** If the p-value from the ADF test is less than a chosen significance level (e.g., 0.05), you reject $H_0$ and conclude the series is stationary. If the p-value is greater than 0.05, you fail to reject $H_0$ and conclude the series is non-stationary.

- **Kwiatkowski-Phillips-Schmidt-Shin (KPSS) Test:**
    - Has the opposite null hypothesis to the ADF test.
    - Null Hypothesis ($H_0$): The series is stationary (or stationary around a deterministic trend).
    - Alternative Hypothesis ($H_a$): The series has a unit root (it is non-stationary).
    - **Interpretation:** If the p-value is less than 0.05, you reject $H_0$ and conclude the series is non-stationary. If the p-value is greater than 0.05, you fail to reject $H_0$ and conclude the series is stationary.

It is often good practice to use both ADF and KPSS tests. If they agree, you have stronger evidence.

#### D. Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) Plots

This is crucial for understanding ARIMA.

- **Autocorrelation Function (ACF):**  
    The ACF measures the correlation between a time series $Y_t$ and its lagged values $Y_{t-k}$.  
    The autocorrelation at lag $k$ is defined as:

    $$
    \rho_k = \frac{\mathrm{Cov}(Y_t, Y_{t-k})}{\mathrm{Var}(Y_t)} = \frac{\gamma_k}{\gamma_0}
    $$

    An ACF plot shows $\rho_k$ on the y-axis against the lag $k$ on the x-axis. It typically includes confidence bands (e.g., 95% confidence intervals). If an autocorrelation bar extends beyond these bands, it's considered statistically significant.

- **Partial Autocorrelation Function (PACF):**  
    The PACF measures the correlation between $Y_t$ and $Y_{t-k}$ after removing the linear effects of the intermediate lags ($Y_{t-1}, Y_{t-2}, ..., Y_{t-k+1}$).

---

### How ACF/PACF Plots Indicate Stationarity (or Lack Thereof)

- **Stationary Series:**
    - **ACF:** For a stationary series, the ACF will typically drop to zero (or within the confidence bands) relatively quickly.
        - For an AR($p$) process, the ACF decays exponentially or with a damped sine wave pattern.
        - For an MA($q$) process, the ACF cuts off sharply after lag $q$ (becomes zero or statistically insignificant).
    - **PACF:**
        - For an AR($p$) process, the PACF cuts off sharply after lag $p$.
        - For an MA($q$) process, the PACF decays exponentially or with a damped sine wave pattern.

- **Non-Stationary Series (due to Trend / Unit Root):**
    - **ACF:** The ACF will decay very slowly towards zero. The autocorrelations will remain large and positive for many lags.
        - *Why?* For a series with an upward trend, today's value ($Y_t$) is likely to be higher than yesterday's ($Y_{t-1}$), which was higher than the day before, and so on. This creates strong positive correlations that persist across many lags because all these points are part of the same upward movement. The "memory" of the series is very long because of the trend. If $Y_t$ is high, $Y_{t-k}$ (even for large $k$) also tends to be high (relative to the overall mean if it were constant, which it isn't).
    - **PACF:** Often, the PACF will have a significant spike at lag 1 and then cut off (or decay much faster than the ACF). This is characteristic of a random walk or a process close to it.

- **Non-Stationary Series (due to Seasonality):**
    - **ACF:** You will see significant spikes at the seasonal lags (e.g., at lag 12, 24, 36 for monthly data with yearly seasonality). These spikes will decay very slowly (or not at all) at multiples of the seasonal period.
    - **PACF:** May also show spikes at seasonal lags.

---

**This is a critical connection:**  
A slowly decaying ACF is a hallmark of non-stationarity, often due to a trend or unit root. This tells you that the series needs to be transformed (e.g., differenced) to achieve stationarity before you can effectively model its remaining correlation structure with AR or MA terms.


### How the ACF Plot Helps Detect Stationarity

The **Autocorrelation Function (ACF) plot** is a key visual tool for diagnosing stationarity in time series analysis.

#### What is the ACF Plot?
- The ACF plot shows the correlation of a time series with its own past values (lags).
- For each lag $k$, it plots the correlation coefficient between $y_t$ and $y_{t-k}$.

#### How to Interpret the ACF for Stationarity

- **Stationary Series:**
    - The ACF drops to near zero quickly (after a few lags).
    - There are no strong, persistent patterns.
    - The plot shows only a few significant spikes close to lag 0, with the rest within the confidence bounds.

- **Non-Stationary Series:**
    - The ACF decays slowly, often remaining high for many lags.
    - This slow decay indicates the presence of trend or strong seasonality.
    - The plot may show a gradual decline or a sinusoidal pattern, suggesting non-stationarity.

#### Typical Patterns

| Series Type         | ACF Pattern                                   |
|---------------------|-----------------------------------------------|
| Stationary          | Rapid drop-off after lag 0                    |
| Trend (non-stationary) | Slow, gradual decay                        |
| Seasonal            | Significant spikes at seasonal lags           |

#### Example

- If your ACF plot shows a slow decay, your series is likely **non-stationary** and may need differencing.
- If the ACF cuts off quickly, your series is likely **stationary**.

#### Practical Steps

1. **Plot the ACF of your series.**
2. **Look for slow decay:** Indicates non-stationarity.
3. **Look for rapid drop-off:** Indicates stationarity.
4. **After differencing:** Re-plot the ACF to check if stationarity is achieved.

---

**In summary:**  
A stationary time series will have an ACF that quickly drops to zero, while a non-stationary series will have an ACF that decays slowly. This visual cue helps you decide if further transformations (like differencing) are needed.

ARIMA models are designed to capture autocorrelation (linear dependence) in stationary time series. If the ACF drops quickly to zero, it means the series is already close to white noise—there is little to no linear relationship between current and past values.

**How ARIMA Works:**
- ARIMA fits a linear model where the current value is regressed on its own past values (AR), past forecast errors (MA), and possibly differences (I).
- The model is most effective when the series is stationary but still shows autocorrelation (i.e., ACF/PACF have significant spikes at some lags).

**If ACF Drops Quickly to Zero:**
- The series is essentially white noise—future values are unpredictable from past values.
- In this case, ARIMA will estimate all AR and MA coefficients as zero (or close), reducing to a simple mean model.
- There is nothing left for ARIMA to model; forecasts will be flat (the mean).

**Summary Table:**

| ACF Pattern         | What ARIMA Models?                |
|---------------------|-----------------------------------|
| Slow decay          | Trend/seasonality (needs differencing) |
| Significant spikes  | AR/MA structure (good for ARIMA)  |
| Drops to zero fast  | White noise (nothing to model)    |

**Key Point:**  
ARIMA is powerful when the stationary series still has autocorrelation. If not, ARIMA cannot improve on a naive mean forecast.

Yes, the KPSS test is appropriate for electricity load demand data, even when there is no trend.

- **KPSS Test Purpose:** It tests the null hypothesis that a time series is stationary (either level-stationary or trend-stationary, depending on the test version).
- **For No Trend:** If your electricity load data has no trend, use the KPSS test with the "level" (default) option. This checks for level stationarity.
- **Interpretation:**  
    - If the p-value is high (e.g., > 0.05), you fail to reject the null hypothesis and conclude the series is stationary.
    - If the p-value is low, the series is likely non-stationary.

**Summary:**  
The KPSS test is valid and useful for checking stationarity in electricity load demand data, regardless of whether a trend is present. Just ensure you use the correct version of the test (level vs. trend) based on your data characteristics.



### What to Do When ADF and KPSS Give Conflicting Results?

It is not uncommon for the Augmented Dickey-Fuller (ADF) and KPSS tests to give **conflicting results** regarding stationarity:

- **ADF rejects unit root (p-value < 0.05):**  
    Suggests the series is stationary.
- **KPSS rejects stationarity (p-value < 0.05):**  
    Suggests the series is non-stationary.

#### Why Does This Happen?

- **Different Null Hypotheses:**  
    - ADF: $H_0$ = non-stationary (unit root present), $H_a$ = stationary.
    - KPSS: $H_0$ = stationary, $H_a$ = non-stationary (unit root present).
- **Sensitivity to Model Specification:**  
    - Both tests can be sensitive to trend, seasonality, or structural breaks.
    - If the series is *trend-stationary* (stationary after removing a deterministic trend), ADF may reject the unit root, but KPSS may still reject stationarity if the trend is not properly modeled.
- **Finite Sample Effects:**  
    - Both tests can have low power or size distortions in small samples or with strong autocorrelation.

#### What Should You Do?

1. **Visually Inspect the Series:**  
    - Plot the time series and its ACF/PACF.
    - Look for obvious trends, seasonality, or structural breaks.

2. **Check Model Specification:**  
    - Did you include a trend term in the tests if the data has a trend?
    - Try running ADF/KPSS with and without trend terms.

3. **Consider the Nature of Non-Stationarity:**  
    - If the series is *trend-stationary*, detrending (removing the trend) may be more appropriate than differencing.
    - If the series is *difference-stationary*, differencing is needed.

4. **Try Additional Tests:**  
    - Use other tests (e.g., Phillips-Perron, Zivot-Andrews for structural breaks) for more evidence.

5. **Practical Approach:**  
    - If in doubt, apply a transformation (differencing or detrending), then re-test.
    - Choose the simplest transformation that achieves stationarity (as confirmed by both tests and visual inspection).

#### Example Table

| Scenario                          | ADF Result | KPSS Result | Likely Situation         | Suggested Action                |
|------------------------------------|------------|-------------|-------------------------|---------------------------------|
| Both stationary                    | Reject     | Do not reject | Stationary             | Proceed with modeling           |
| Both non-stationary                | Do not reject | Reject     | Non-stationary         | Difference or detrend           |
| ADF rejects, KPSS rejects          | Reject     | Reject      | Trend-stationary or structural break | Try detrending, check for breaks |
| ADF does not reject, KPSS does not reject | Do not reject | Do not reject | Inconclusive/Borderline | Gather more evidence, inspect visually |

#### Key Takeaway

> **Always combine statistical tests with visual inspection and domain knowledge.**  
> If ADF and KPSS disagree, check for trends, seasonality, or breaks, and try both differencing and detrending. The goal is to achieve a stationary series for reliable modeling.

In [ ]:
fig = plot_acf(original)
adf_stat, adf_pvalue, _, _, _, _ = adfuller(original)
print(f"ADF stat: {adf_stat:.3f}, p-value: {adf_pvalue:.3f}")

kpss_stat, kpss_pvalue, _, _ = kpss(original)
print(f"kpss_stat: {kpss_stat:.3f}, kpss_pvalue: {kpss_pvalue:.2f}")
fig.add_annotation(
    text=f"ADF p-value: {adf_pvalue:.3f}<br>KPSS p-value: {kpss_pvalue:.3f}",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.2,
    showarrow=False,
    align="center",
)
fig.show()

The results from the ADF and KPSS tests both show extremely small p-values (close to $0$), which provides strong statistical evidence that the original series is **not stationary**. Specifically:

- **ADF p-value $\approx 0$:** Rejects the null hypothesis of a unit root, but in practice, such a low value (along with the KPSS result) often indicates the presence of strong seasonality or structural non-stationarity that the test is picking up.
- **KPSS p-value $\approx 0$:** Rejects the null hypothesis of stationarity, confirming non-stationarity.

This pattern—where both tests reject their respective nulls—typically suggests the series is **difference-stationary**: it can be made stationary by differencing.

Looking at the ACF plot, we see that autocorrelation decays slowly toward zero, and there are significant spikes at lags greater than $40$. This is a classic sign of non-stationarity with strong **seasonal components** (e.g., daily or weekly cycles in half-hourly data). Therefore, differencing (possibly both regular and seasonal) is required to achieve stationarity before fitting ARIMA models.

In [ ]:
fig = plot_acf(diff)
adf_stat, adf_pvalue, _, _, _, _ = adfuller(diff)
print(f"ADF stat: {adf_stat:.3f}, p-value: {adf_pvalue:.3f}")

kpss_stat, kpss_pvalue, _, _ = kpss(diff)
print(f"kpss_stat: {kpss_stat:.3f}, kpss_pvalue: {kpss_pvalue:.2f}")
fig.add_annotation(
    text=f"ADF p-value: {adf_pvalue:.3f}<br>KPSS p-value: {kpss_pvalue:.3f}",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.2,
    showarrow=False,
    align="center",
)
fig.show()

In [ ]:
fig = plot_acf(diff_day_diff)
adf_stat, adf_pvalue, _, _, _, _ = adfuller(diff_day_diff)
print(f"ADF stat: {adf_stat:.3f}, p-value: {adf_pvalue:.3f}")

kpss_stat, kpss_pvalue, _, _ = kpss(diff_day_diff)
print(f"kpss_stat: {kpss_stat:.3f}, kpss_pvalue: {kpss_pvalue:.2f}")
fig.add_annotation(
    text=f"ADF p-value: {adf_pvalue:.3f}<br>KPSS p-value: {kpss_pvalue:.3f}",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.2,
    showarrow=False,
    align="center",
)
fig.show()

In [ ]:
fig = plot_acf(day_diff)
adf_stat, adf_pvalue, _, _, _, _ = adfuller(day_diff)
print(f"ADF stat: {adf_stat:.3f}, p-value: {adf_pvalue:.3f}")

kpss_stat, kpss_pvalue, _, _ = kpss(day_diff)
print(f"kpss_stat: {kpss_stat:.3f}, kpss_pvalue: {kpss_pvalue:.2f}")
fig.add_annotation(
    text=f"ADF p-value: {adf_pvalue:.3f}<br>KPSS p-value: {kpss_pvalue:.3f}",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.2,
    showarrow=False,
    align="center",
)
fig.show()

After applying daily differencing, the ACF plot now shows a much more rapid decay toward zero, with only a slight spike remaining around lag $45$ (close to the daily period of $48$). This indicates that much of the non-stationarity—particularly some of the strong seasonality—has been removed from the series.

Importantly, both the ADF and KPSS tests now agree: their p-values suggest that the differenced series is **stationary**. This agreement between visual diagnostics (ACF) and formal statistical tests provides strong evidence that the series is now suitable for ARIMA modeling. The small spike near lag $45$ may reflect residual daily seasonality or minor periodic effects, but the overall structure is now much closer to a stationary process.

Even after applying daily differencing with a lag of $48$ (which corresponds to one day in half-hourly data), we may still observe small but noticeable spikes in the ACF plot near the daily lag (e.g., around lag $45$ or $48$). This suggests that **some residual daily seasonality remains** in the series, even though the main seasonal component has been largely removed.

Why does this happen? High-frequency time series data—like half-hourly electricity demand—often exhibit **complex or "imperfect" seasonality**. Here are some reasons why daily differencing might not fully eliminate daily patterns:

- **Non-integer or drifting seasonality:** Real-world daily cycles may not align perfectly with a fixed period of $48$ time steps, especially due to factors like daylight saving time, holidays, or behavioral changes across days.
- **Changing daily patterns:** The shape of the daily cycle can vary between weekdays and weekends, or across seasons, so a single differencing period cannot capture all variations.
- **Interactions with other seasonalities:** If there is also strong weekly seasonality (period $336$), the interaction between daily and weekly cycles can leave behind residual patterns after differencing at just one period.

**In summary:**  
Daily differencing ($s=48$) is effective at removing the dominant daily cycle, but high-frequency data often contains more subtle or overlapping seasonal effects. These can leave behind small autocorrelation spikes, indicating that the series is not perfectly stationary with respect to all forms of daily seasonality. In practice, this is common and usually not a major concern for ARIMA modeling, as long as the remaining autocorrelations are weak and the statistical tests (ADF, KPSS) indicate stationarity. If strong residual seasonality persists, consider additional seasonal differencing, using models that handle multiple seasonalities (like TBATS or Prophet), or including exogenous variables to capture complex patterns.

In [ ]:
fig = plot_acf(week_diff)
adf_stat, adf_pvalue, _, _, _, _ = adfuller(week_diff)
print(f"ADF stat: {adf_stat:.3f}, p-value: {adf_pvalue:.3f}")

kpss_stat, kpss_pvalue, _, _ = kpss(week_diff)
print(f"kpss_stat: {kpss_stat:.3f}, kpss_pvalue: {kpss_pvalue:.2f}")
fig.add_annotation(
    text=f"ADF p-value: {adf_pvalue:.3f}<br>KPSS p-value: {kpss_pvalue:.3f}",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.2,
    showarrow=False,
    align="center",
)
fig.show()

After applying weekly differencing ($s=336$), the time series appears more stationary, as indicated by a faster decay in the ACF and improved results from the stationarity tests. However, strong daily seasonality ($s=48$) is still clearly visible in both the plot and the ACF, with significant spikes at daily lags. This outcome is expected: weekly differencing effectively removes the weekly pattern, but leaves the daily cycle largely intact.

 In practice, this means that while the series is closer to stationarity, further transformation—such as daily differencing or using models that can handle multiple seasonalities—may be necessary to fully achieve stationarity for ARIMA modeling.

In [ ]:
fig = plot_acf(day_week_diff)
adf_stat, adf_pvalue, _, _, _, _ = adfuller(day_week_diff)
print(f"ADF stat: {adf_stat:.3f}, p-value: {adf_pvalue:.3f}")

kpss_stat, kpss_pvalue, _, _ = kpss(day_week_diff)
print(f"kpss_stat: {kpss_stat:.3f}, kpss_pvalue: {kpss_pvalue:.2f}")
fig.add_annotation(
    text=f"ADF p-value: {adf_pvalue:.3f}<br>KPSS p-value: {kpss_pvalue:.3f}",
    xref="paper",
    yref="paper",
    x=0.5,
    y=-0.2,
    showarrow=False,
    align="center",
)
fig.show()

With both daily and weekly differencing applied, the ACF plot for the first $48$ periods closely resembles the ACF after just daily differencing. This is expected: since we are only displaying lags up to $48$, we primarily observe the effects of daily seasonality. Weekly differencing ($s=336$) mainly impacts autocorrelation at much larger lags (multiples of $336$), which are not visible in this plot. Therefore, for short lags, the ACF pattern is dominated by the daily differencing, and additional weekly differencing does not substantially alter the appearance within the first $48$ lags. This highlights the importance of choosing an appropriate lag range when interpreting ACF plots for series with multiple seasonalities.

## 7. Plot ACF and PACF

Plot ACF and PACF for original and differenced series to help identify ARIMA orders.

---

### Understanding ACF and PACF for ARIMA Order Selection

When building ARIMA models, one of the most important steps is to determine the appropriate values for the AR (autoregressive), I (integrated/differencing), and MA (moving average) terms. The ACF (Autocorrelation Function) and PACF (Partial Autocorrelation Function) plots are essential diagnostic tools for this purpose.

#### **What Are ACF and PACF?**

- **ACF (Autocorrelation Function):**  
    Measures the correlation between the time series and its own lagged values. For each lag $k$, it shows how strongly $y_t$ is correlated with $y_{t-k}$.
- **PACF (Partial Autocorrelation Function):**  
    Measures the correlation between $y_t$ and $y_{t-k}$ after removing the effects of all shorter lags ($1, 2, ..., k-1$). It helps isolate the direct relationship at each lag.

#### **How Do ACF and PACF Help Identify ARIMA Orders?**

Suppose you have already differenced your series to achieve stationarity (as discussed in previous sections). Now, you want to determine the AR and MA orders:

- **AR($p$):** Number of autoregressive terms (how many past values to use)
- **MA($q$):** Number of moving average terms (how many past forecast errors to use)

##### **Typical Patterns in ACF and PACF**

| Model Type | ACF Pattern | PACF Pattern | What to Look For |
|------------|-------------|--------------|------------------|
| AR($p$)    | Tails off (decays gradually) | Cuts off after lag $p$ | Significant spikes in PACF up to lag $p$, then drops to zero |
| MA($q$)    | Cuts off after lag $q$ | Tails off (decays gradually) | Significant spikes in ACF up to lag $q$, then drops to zero |
| ARMA($p,q$)| Both tail off | Both tail off | No sharp cut-off; both decay gradually |

- **"Cuts off"** means the plot shows significant spikes up to a certain lag, then all subsequent lags are within the confidence bounds (not significant).
- **"Tails off"** means the plot decays slowly, with no clear cut-off.

##### **Example:**

- If the **PACF** plot shows significant spikes at lags 1 and 2, then drops to zero, while the **ACF** decays gradually, this suggests an **AR(2)** model.
- If the **ACF** plot shows significant spikes at lags 1 and 2, then drops to zero, while the **PACF** decays gradually, this suggests an **MA(2)** model.
- If both ACF and PACF tail off, an **ARMA** model (with both AR and MA terms) may be appropriate.

##### **Seasonal ARIMA (SARIMA):**

For seasonal data, look for significant spikes at seasonal lags (e.g., lag 48 for daily seasonality in half-hourly data). The same logic applies, but at seasonal lags.

---

### **Step-by-Step: How to Use ACF and PACF**

1. **Plot the ACF and PACF of your (differenced) series.**
2. **Look for cut-offs and tailing patterns:**
     - Where do the significant spikes end?
     - Does the plot decay slowly or drop off quickly?
3. **Decide on $p$ and $q$ based on the patterns above.**
4. **For seasonal data, repeat the process at seasonal lags to determine seasonal AR and MA orders.**

---

### **Summary Table**

| Plot Feature | Model Suggestion |
|--------------|-----------------|
| PACF cuts off at lag $p$, ACF tails off | AR($p$) |
| ACF cuts off at lag $q$, PACF tails off | MA($q$) |
| Both tail off | ARMA($p,q$) |
| Spikes at seasonal lags | Add seasonal AR or MA terms |

---

By carefully examining the ACF and PACF plots, you can make informed choices about the ARIMA model structure, leading to better forecasts and more interpretable models. Always combine these diagnostics with model selection criteria (like AIC/BIC) and out-of-sample validation for best results.

In [ ]:
plot_series_acf_pacf(
    data=original,
    time=data.get_column(time_),
)

In [ ]:
plot_series_acf_pacf(
    data=diff_day_diff,
)

In [ ]:
plot_series_acf_pacf(
    data=day_diff,
)

In [ ]:
plot_series_acf_pacf(
    data=day_week_diff,
)

In [ ]:
plot_series(
    df=data.with_columns(target_col.diff(48).diff(336)).drop_nulls(),
    date_range=["2012-11-4", "2012-12-4"],
)

## 8. Determine Differencing Orders

Use `ndiffs` and `nsdiffs` to determine the required number of differences for stationarity and seasonality.

---

### Understanding `ndiffs` and `nsdiffs`: How to Identify Required Differencing Orders

When preparing a time series for ARIMA modeling, one of the most important steps is to determine **how many times to difference the series** to achieve stationarity. This includes both:

- **Regular (non-seasonal) differencing**: Removes trends.
- **Seasonal differencing**: Removes repeating seasonal patterns.

#### Why Do We Need Differencing?

Recall that ARIMA models assume the input series is **stationary**—its mean, variance, and autocorrelation structure do not change over time. Most real-world time series (like electricity demand) are not stationary due to trends and seasonality. Differencing is the main tool to transform such series into stationary ones.

---

### What Are `ndiffs` and `nsdiffs`?

- **`ndiffs`**: Estimates the minimum number of **regular differences** ($d$) needed to make a series stationary.
- **`nsdiffs`**: Estimates the minimum number of **seasonal differences** ($D$) needed to remove seasonal non-stationarity, given a seasonal period $s$.

Both functions are available in the `statsforecast` package (and similar ones in `pmdarima` and `forecast` in R).

---

### How Do `ndiffs` and `nsdiffs` Work?

#### 1. `ndiffs`

- **Purpose**: Find the smallest integer $d$ such that the $d$-th differenced series is stationary.
- **How?**: It applies a statistical test (like the Augmented Dickey-Fuller, KPSS, or Phillips-Perron test) to the original and successively differenced series.
    - If the test suggests non-stationarity, it differences the series again and repeats.
    - Stops when the test indicates stationarity or a maximum number of differences is reached (usually 2).

**Example:**
- If `ndiffs(series)` returns $1$, you should difference the series once ($y'_t = y_t - y_{t-1}$) before fitting ARIMA.

#### 2. `nsdiffs`

- **Purpose**: Find the smallest integer $D$ such that the $D$-th **seasonally differenced** series (with period $s$) is stationary.
- **How?**: It applies a seasonal stationarity test (like the OCSB or Canova-Hansen test) to the original and seasonally differenced series.
    - If the test suggests seasonal non-stationarity, it differences the series at lag $s$ and repeats.
    - Stops when the test indicates seasonal stationarity or a maximum number of seasonal differences is reached (usually 1 or 2).

**Example:**
- If `nsdiffs(series, period=48)` returns $1$, you should apply one seasonal difference at lag 48 ($y'_t = y_t - y_{t-48}$).

---

### How to Use `ndiffs` and `nsdiffs` in Practice

Suppose you have a half-hourly electricity demand series with strong daily ($s=48$) and weekly ($s=336$) seasonality.

1. **Check Regular Differencing:**
    ```python
    d = ndiffs(original.to_numpy())
    print(f"Recommended number of regular differences (d): {d}")
    ```
    - If $d=1$, difference the series once.

2. **Check Seasonal Differencing:**
    ```python
    D_daily = nsdiffs(original.to_numpy(), period=48)
    print(f"Recommended number of daily seasonal differences (D_daily): {D_daily}")

    D_weekly = nsdiffs(original.to_numpy(), period=336)
    print(f"Recommended number of weekly seasonal differences (D_weekly): {D_weekly}")
    ```
    - If $D_{daily}=1$, apply one seasonal difference at lag 48.
    - If $D_{weekly}=1$, apply one seasonal difference at lag 336.

3. **Apply Differences as Needed:**
    - Start with the recommended seasonal difference(s), then regular difference(s).
    - After each step, check stationarity using ACF plots and statistical tests (ADF, KPSS).

---

### Example Workflow

Suppose the outputs are:
- `ndiffs(original) = 0`
- `nsdiffs(original, period=48) = 1`
- `nsdiffs(original, period=336) = 0`

**Interpretation:**
- No regular differencing needed ($d=0$).
- One seasonal difference at lag 48 needed ($D=1$).
- No weekly seasonal difference needed.

**You would difference the series at lag 48:**
$$
y'_t = y_t - y_{t-48}
$$

---

### Key Points

- **`ndiffs` and `nsdiffs` automate the process of identifying how much differencing is needed.**
- **Always confirm with visual inspection and stationarity tests after differencing.**
- **Over-differencing can introduce unnecessary noise; use the minimum differencing needed.**

---

By using `ndiffs` and `nsdiffs`, you can systematically and objectively determine the differencing orders required to make your time series stationary, setting a solid foundation for effective ARIMA modeling.

In [ ]:
original.is_null().any()

In [ ]:
ndiffs(original.to_numpy())

In [ ]:
nsdiffs(original.to_numpy(), period=48, test="kpss")

In [ ]:
nsdiffs(original.to_numpy(), period=48 * 7)

## 9. Fit ARIMA Model

Configure and fit an ARIMA model using StatsForecast with specified orders and seasonal parameters.

## What is the ARIMA Model? A Beginner-Friendly Explanation

The **ARIMA** model is one of the most widely used and powerful statistical models for time series forecasting. Its name stands for:

- **A**uto**R**egressive (**AR**)
- **I**ntegrated (**I**)
- **M**oving **A**verage (**MA**)

Let's break down each component and see how they work together to model time series data.

---

### 1. The Building Blocks of ARIMA

#### **A. Autoregressive (AR) Part**

- The AR part models the relationship between the current value and its own previous values (lags).
- **Mathematically:**  
    $$
    y_t = \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \varepsilon_t
    $$
    where:
    - $y_t$ is the value at time $t$
    - $\phi_1, \ldots, \phi_p$ are coefficients
    - $p$ is the number of lags (the AR order)
    - $\varepsilon_t$ is white noise (random error)

#### **B. Integrated (I) Part**

- The I part refers to **differencing** the series to make it stationary (i.e., removing trends or seasonality).
- **First-order differencing:**  
    $$
    y'_t = y_t - y_{t-1}
    $$
- The number of times you difference the series is called the **order of integration** ($d$).

#### **C. Moving Average (MA) Part**

- The MA part models the relationship between the current value and past forecast errors (residuals).
- **Mathematically:**  
    $$
    y_t = \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \cdots + \theta_q \varepsilon_{t-q} + \varepsilon_t
    $$
    where:
    - $\theta_1, \ldots, \theta_q$ are coefficients
    - $q$ is the number of lagged forecast errors (the MA order)

---

### 2. The Full ARIMA Model

The full ARIMA($p$, $d$, $q$) model combines all three components:

- $p$: Number of autoregressive terms (AR order)
- $d$: Number of differences needed to make the series stationary (Integration order)
- $q$: Number of moving average terms (MA order)

**General formula (after differencing $d$ times):**
$$
y'_t = \phi_1 y'_{t-1} + \cdots + \phi_p y'_{t-p} + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q} + \varepsilon_t
$$

---

### 3. Why Use ARIMA?

- **Handles Trend and Seasonality:** By differencing, ARIMA can model non-stationary data with trends or seasonal patterns.
- **Flexible:** Can be extended to seasonal ARIMA (SARIMA) for more complex seasonality.
- **Widely Used:** Standard tool in forecasting electricity demand, sales, finance, and more.

---

### 4. How to Build an ARIMA Model (Step-by-Step)

1. **Visualize and Explore the Data:**  
   Plot the series, look for trends and seasonality.

2. **Make the Series Stationary:**  
   Apply differencing as needed (the "I" part).

3. **Identify AR and MA Orders:**  
   Use ACF and PACF plots to choose $p$ and $q$.

4. **Fit the Model:**  
   Estimate the coefficients using historical data.

5. **Diagnose Residuals:**  
   Check if the residuals look like white noise (no pattern left).

6. **Forecast:**  
   Use the model to predict future values.

---

### 5. Example

Suppose you have half-hourly electricity demand data:

- You difference the series once to remove trend ($d=1$).
- The PACF plot suggests 2 significant lags ($p=2$).
- The ACF plot suggests 1 significant lag ($q=1$).

You would fit an **ARIMA(2, 1, 1)** model.

---

### 6. Key Points

- **ARIMA models are best for stationary time series** (constant mean and variance).
- **Differencing** is used to make the series stationary.
- **ACF and PACF plots** help you choose the AR and MA orders.
- **Extensions:**  
    - **SARIMA** adds seasonal terms for data with strong seasonality.
    - **ARIMAX/SARIMAX** can include exogenous variables (external predictors).

---

### 7. Summary Table

| Component | What it Does                | Parameter |
|-----------|----------------------------|-----------|
| AR        | Uses past values            | $p$       |
| I         | Differences the series      | $d$       |
| MA        | Uses past forecast errors   | $q$       |

---

**In summary:**  
The ARIMA model is a powerful, flexible tool for forecasting time series data, especially when you carefully prepare your data and choose the right parameters!

### How Does SARIMA Work? Can It Handle Multiple Seasonalities?

#### What is SARIMA?

**SARIMA** stands for **Seasonal AutoRegressive Integrated Moving Average**. It extends the classic ARIMA model to handle time series data with **seasonal patterns**—that is, data where values repeat at regular intervals (like daily, weekly, or yearly cycles).

The SARIMA model is denoted as:

$$
\text{SARIMA}(p, d, q) \times (P, D, Q)_s
$$

where:
- $p, d, q$ are the non-seasonal ARIMA orders (autoregressive, differencing, moving average)
- $P, D, Q$ are the **seasonal** ARIMA orders (seasonal AR, seasonal differencing, seasonal MA)
- $s$ is the **seasonal period** (e.g., $s=12$ for monthly data with yearly seasonality, $s=48$ for half-hourly data with daily seasonality)

#### How Does SARIMA Work?

SARIMA combines both **non-seasonal** and **seasonal** components:

- **Non-seasonal part:** Models short-term dependencies and trends, just like ARIMA.
- **Seasonal part:** Models repeating patterns at a fixed period $s$.

The general SARIMA model equation (after differencing) is:

$$
\Phi_P(B^s) \phi_p(B) (1 - B)^d (1 - B^s)^D y_t = \Theta_Q(B^s) \theta_q(B) \varepsilon_t
$$

where:
- $B$ is the backshift operator ($B y_t = y_{t-1}$)
- $\phi_p(B)$ and $\theta_q(B)$ are non-seasonal AR and MA polynomials
- $\Phi_P(B^s)$ and $\Theta_Q(B^s)$ are seasonal AR and MA polynomials at lag $s$
- $(1 - B)^d$ is regular differencing, $(1 - B^s)^D$ is seasonal differencing

**In plain English:**  
SARIMA fits both regular ARIMA terms and additional AR/MA terms at the seasonal lag $s$ to capture repeating cycles.

#### Can SARIMA Handle Multiple Seasonalities?

**Standard SARIMA can only handle one seasonal period at a time.**

- For example, you can model **daily** seasonality ($s=48$ for half-hourly data) or **weekly** seasonality ($s=336$), but not both simultaneously in the same SARIMA model.
- If your data has **multiple strong seasonalities** (e.g., both daily and weekly cycles), standard SARIMA is not sufficient.

##### Why Not?

- The SARIMA model structure only allows for one set of seasonal AR, MA, and differencing terms at a single period $s$.
- Real-world data (like electricity demand) often has **multiple seasonal cycles** (e.g., daily and weekly), which interact in complex ways.

#### What If You Have Multiple Seasonalities?

For time series with **multiple seasonalities**, you need more advanced models, such as:

- **TBATS** (Trigonometric, Box-Cox, ARMA errors, Trend, and Seasonal components): Handles multiple and non-integer seasonalities.
- **Prophet** (by Facebook): Can model multiple seasonalities and holiday effects.
- **Dynamic Harmonic Regression**: Uses Fourier terms to capture multiple seasonal cycles.
- **Multiple Seasonal ARIMA (MSARIMA)**: Some specialized implementations exist, but not in standard libraries.

#### Practical Example

- **SARIMA:** Good for data with a single dominant seasonality (e.g., only daily or only yearly).
- **TBATS/Prophet:** Better for data with both daily and weekly cycles (like half-hourly electricity demand).

#### Summary Table

| Model   | Handles Trend | Handles 1 Seasonality | Handles Multiple Seasonalities |
|---------|:-------------:|:--------------------:|:-----------------------------:|
| ARIMA   |      ✓        |          ✗           |              ✗                |
| SARIMA  |      ✓        |          ✓           |              ✗                |
| TBATS   |      ✓        |          ✓           |              ✓                |
| Prophet |      ✓        |          ✓           |              ✓                |

---

**Key Takeaway:**  
SARIMA is a powerful extension of ARIMA for **single** seasonality, but for **multiple** seasonalities (like daily and weekly), you should use models like TBATS or Prophet.

In [ ]:
fcst = StatsForecast(
    models=[
        ARIMA(
            order=(2, 0, 1),
            seasonal_order=(1, 1, 1),
            season_length=48,
            alias="ARIMA(2,0,1)(1,1,1)[48]",
        ),
        ARIMA(
            order=(1, 0, 3),
            seasonal_order=(1, 1, 1),
            season_length=48,
            alias="ARIMA(1,0,3)(1,1,1)[48]",
        ),
    ],
    freq="30m",
)

## 10. Forecast and Visualize Results

Generate forecasts using cross-validation and plot the forecasted values against the actual series.

In [ ]:
y_hat = fcst.cross_validation(
    df=data.select([id_, time_, target_]).to_pandas(),
    h=48 * 7,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop(columns=["cutoff"])

### Why Does Fitting ARIMA Models Take So Long for 3 Years of Hourly Data?

Fitting ARIMA models to long, high-frequency time series (like 3 years of hourly or half-hourly data) can be **computationally intensive**. Here’s why:

---

#### 1. **Large Number of Data Points**

- **3 years of hourly data:**  
    $3 \text{ years} \times 365 \text{ days/year} \times 24 \text{ hours/day} \approx 26,\!280$ data points.
- **3 years of half-hourly data:**  
    $3 \times 365 \times 48 \approx 52,\!560$ data points.

The more data points, the more calculations are required for each step of model fitting.

---

#### 2. **ARIMA Model Fitting is Iterative and Complex**

- ARIMA fitting involves **maximum likelihood estimation** (MLE), which is an iterative optimization process.
- For each iteration, the algorithm must:
        - Compute the likelihood of the model given the data.
        - Update parameters (AR, MA, seasonal terms) to maximize the likelihood.
        - Recompute residuals and possibly invert large matrices.
- **Seasonal ARIMA** (SARIMA) models add even more parameters and complexity, especially with long seasonal periods (e.g., $s=48$ for daily seasonality in half-hourly data).

---

#### 3. **Memory and Computational Cost Grows with Data Size**

- The time and memory required for ARIMA fitting **increase with the length of the series** and the number of parameters.
- For long series, each likelihood evaluation is slower, and more iterations may be needed for convergence.

---

#### 4. **Why Not Use Auto-ARIMA?**

- **Auto-ARIMA** automatically searches over many combinations of $(p, d, q)$ and seasonal $(P, D, Q)$ parameters.
- For each candidate model, it fits a full ARIMA model (as above), often **hundreds or thousands of times**.
- This makes auto-ARIMA **much slower** than fitting a single, specified ARIMA model—sometimes by an order of magnitude or more.

---

#### 5. **Summary Table**

| Approach         | What It Does                | Speed (for long series) |
|------------------|----------------------------|-------------------------|
| ARIMA (manual)   | Fit one model with chosen orders | Slow (minutes)         |
| Auto-ARIMA       | Search and fit many models      | Very slow (tens of minutes or hours) |

---

#### 6. **Practical Tips**

- For long, high-frequency series, **specify ARIMA orders manually** (using ACF/PACF and domain knowledge) to save time.
- Consider **downsampling** (e.g., aggregate to hourly or daily) if fine granularity is not needed.
- Use **faster implementations** (e.g., `statsforecast`, `pmdarima`, or parallel processing) if available.

---

**In summary:**  
Fitting ARIMA models on long, high-frequency time series is slow because of the large number of data points and the complexity of the estimation process. Auto-ARIMA is even slower because it fits many models. For practical workflows, it’s often best to choose model orders manually for large datasets.

In [ ]:
plot_series(data, pl.from_pandas(y_hat), max_insample_length=48 * 7)

## 11. Analyze Residuals

Extract residuals from the fitted model, plot diagnostics, and run Ljung-Box test on residuals.

In [ ]:
fitted_values = fcst.cross_validation_fitted_values()
insample_forecasts = fitted_values["ARIMA"]
residuals = fitted_values["y"] - insample_forecasts

In [ ]:
plot_residuals_diagnostic(
    residuals=residuals,
    time=fitted_values["ds"],
)

In [ ]:
ljung_box = acorr_ljungbox(residuals, lags=[10], model_df=5)
ljung_box

## 12. Evaluate Forecast Performance

Compute and display forecast accuracy metrics such as MAE, MSE, RMSE, MAPE, SMAPE, and MASE.

In [ ]:
metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48 * 7),
]
evaluate(
    pl.from_pandas(y_hat),
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

## 13. Conclusion: Stationarity, Testing, and ARIMA as a Baseline

---

### **Key Takeaways from This Notebook**

#### **1. Stationarity is Essential for Time Series Modeling**

- **Stationarity** means that a time series has constant mean, variance, and autocorrelation structure over time.
- Most classical time series models, including ARIMA, require the input series to be stationary.
- We achieved stationarity through **differencing** (both regular and seasonal), as confirmed by visual inspection, ACF plots, and formal tests (ADF and KPSS).

#### **2. Statistical Tests Help Diagnose Stationarity**

- The **Augmented Dickey-Fuller (ADF)** test checks for unit roots (non-stationarity due to trend).
- The **KPSS** test checks for stationarity (null hypothesis).
- Using both tests together, along with ACF/PACF plots, provides a robust framework for diagnosing and confirming stationarity.

#### **3. ARIMA Model Fitting: Strengths and Limitations**

- **ARIMA** models are powerful for capturing linear dependencies in stationary time series.
- They are interpretable, widely used, and serve as a strong **baseline** for forecasting tasks.
- In this notebook, we fit SARIMA models to half-hourly electricity demand data, using careful order selection based on ACF/PACF and differencing guided by `ndiffs` and `nsdiffs`.

#### **4. Why ARIMA is a Good Baseline, But Not Always the Best for High-Frequency Data**

- **Strengths:**
    - ARIMA provides a solid, interpretable starting point for time series forecasting.
    - It is effective for data with a single dominant seasonality and no complex nonlinearities.

- **Limitations for High-Frequency, Multi-Seasonal Data:**
    - **Multiple Seasonalities:** Standard ARIMA/SARIMA can only handle one seasonal period at a time (e.g., daily or weekly, but not both). High-frequency data like electricity demand often exhibits both daily and weekly cycles, as well as holiday effects and other complexities.
    - **Complex Patterns:** Real-world high-frequency data may have nonlinearities, time-varying seasonal effects, and interactions that ARIMA cannot capture.
    - **Computational Cost:** Fitting ARIMA models to long, high-frequency series is computationally intensive, especially with auto-ARIMA.

- **Better Alternatives for Complex Seasonality:**
    - Models like **TBATS**, **Prophet**, or **Dynamic Harmonic Regression** are designed to handle multiple and non-integer seasonalities.
    - These models can flexibly capture the rich structure present in high-frequency time series.

---

### **Summary Table**

| Model   | Handles Trend | Handles 1 Seasonality | Handles Multiple Seasonalities | Interpretable | Fast for Long Series |
|---------|:-------------:|:--------------------:|:-----------------------------:|:-------------:|:-------------------:|
| ARIMA   |      ✓        |          ✗           |              ✗                |      ✓        |         ✗           |
| SARIMA  |      ✓        |          ✓           |              ✗                |      ✓        |         ✗           |
| TBATS   |      ✓        |          ✓           |              ✓                |      ✗        |         ✗           |
| Prophet |      ✓        |          ✓           |              ✓                |      ✗        |         ✓           |

---

### **Final Thoughts**

- **ARIMA is an excellent first model** for time series forecasting, especially when you want a transparent, statistically principled baseline.
- For **high-frequency data with multiple seasonalities** (like electricity demand), ARIMA may not fully capture all patterns, and more advanced models should be considered for production forecasting.
- **Always start with careful data exploration, stationarity testing, and baseline modeling before moving to more complex approaches.**

---

By understanding the strengths and limitations of ARIMA, you are well-equipped to build robust forecasting pipelines and to know when to reach for more advanced tools!